# PCA posterior to MINERvA coefficients

The fitted knobs are standardized PCA coordinates $p$. With $C=V\Lambda V^T$ for the MINERvA covariance of $(a_1,\ldots,a_4)$, the inverse transformation is

$$a_{1:4}=a^{\mathrm{MINERvA}}_{1:4}+V\sqrt{\Lambda}\,p.$$

This notebook transforms the **joint MCMC chain**, preserving PCA correlations and non-Gaussian posterior shapes. It starts from the published full MINERvA coefficient vector and propagates constraint-preserving shifts to $a_0,a_5,\ldots,a_8$.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import uproot

# The notebook lives in ma_zexp/python; the shared parametrization is here.
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT.name != 'axial_mass' and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
PARAM_DIR = REPO_ROOT / 'uboone_zexp_reweighting'
if str(PARAM_DIR) not in sys.path:
    sys.path.insert(0, str(PARAM_DIR))

from axial_form_factor_parametrizations import (
    complete_a_values_8, minerva_a_cov_matrix, minerva_a_values, minerva_t0,
)

## Configuration
Change these paths to process a similar PROfit output.

In [ ]:
ROOT_FILE = Path('/nevis/riverside/data/epelaez/ma_zexp/1mu1p_sel/zexp_prior_fits/minerva_k8/minerva_k8_v1_PROfile.root')
OUTPUT_PREFIX = REPO_ROOT / 'ma_zexp/figs/minerva_k8'
BURN_IN = 0
THIN = 1
N_PRIOR_SAMPLES = 100_000

PCA_BRANCHES = [f'weight_spline_FAzexpPCA{i}' for i in range(1, 5)]
PCA_LABELS = [f'FAzexpPCA{i}' for i in range(1, 5)]

## Build the inverse transformation

Let the four free MINERvA coefficients and their covariance be

$$\boldsymbol{a}_{\mathrm{free}}=(a_1,a_2,a_3,a_4)^T,\qquad C_{\mathrm{MINERvA}}=V\Lambda V^T,$$

where the columns of $V$ are the covariance eigenvectors and $\Lambda=\operatorname{diag}(\lambda_1,\ldots,\lambda_4)$. The fitted PROfit variables

$$\boldsymbol{p}=(p_1,p_2,p_3,p_4)^T$$

are standardized PCA coordinates: $p_i=0$ is the MINERvA central value and $p_i=\pm1$ is a one-standard-deviation displacement along eigenvector $i$. Therefore the inverse PCA transformation for the free coefficients is

$$\Delta\boldsymbol{a}_{\mathrm{free}}=B\boldsymbol{p},\qquad B=V\Lambda^{1/2},$$

so that $BB^T=C_{\mathrm{MINERvA}}$.

The full coefficient vector is

$$\boldsymbol{a}=(a_0,a_1,\ldots,a_8)^T.$$

Only $a_1$–$a_4$ are independent. Let $S$ be the $9\times4$ linear map that completes a change in the four free coefficients with changes in $a_0,a_5,\ldots,a_8$ required by $F_A(0)$ and the four sum rules. The complete PCA displacement matrix is then

$$M=SB,$$

where `full_matrix` is this $9\times4$ matrix $M$. Its columns are constructed by applying one unit PCA displacement at a time and subtracting the same completed central vector. The subtraction is important: it makes each column a **homogeneous displacement**, independent of the rounded central coefficients.

Using the published nine-coefficient vector $\boldsymbol{a}_{\mathrm{pub}}$ as the exact prior, the inverse transformation is

$$\boxed{\boldsymbol{a}(\boldsymbol{p})=\boldsymbol{a}_{\mathrm{pub}}+M\boldsymbol{p}}.$$

For the single profile-likelihood best-fit point this appears in code as

$$\boldsymbol{a}_{\mathrm{profile}}=\boldsymbol{a}_{\mathrm{pub}}+M\boldsymbol{p}_{\mathrm{profile}}.$$

For $N$ MCMC draws stored as rows of an $N\times4$ matrix $P$, all posterior samples are transformed at once as

$$A=\boldsymbol{1}_N\boldsymbol{a}_{\mathrm{pub}}^T+PM^T,$$

which explains why the batched NumPy expression uses `pca_samples @ full_matrix.T`. The resulting $A$ has shape $N\times9$, with one complete coefficient vector per row. The corresponding covariance propagation is

$$C_a=M C_p M^T,$$

although the notebook transforms the joint MCMC samples directly so that non-Gaussian posterior structure is retained.

### Constraints preserved by each displacement

For $k_{\max}=8$, the four high-$Q^2$ sum rules are

$$n=0:\quad a_0+a_1+a_2+a_3+a_4+a_5+a_6+a_7+a_8=0,$$
$$n=1:\quad a_1+2a_2+3a_3+4a_4+5a_5+6a_6+7a_7+8a_8=0,$$
$$n=2:\quad 2a_2+6a_3+12a_4+20a_5+30a_6+42a_7+56a_8=0,$$
$$n=3:\quad 6a_3+24a_4+60a_5+120a_6+210a_7+336a_8=0.$$

In addition, $F_A(0)=\sum_{k=0}^8 a_k z(0)^k=-1.2723$. Thus every column $\boldsymbol{m}_j$ of $M$ satisfies all four homogeneous sum rules and $\sum_k (m_j)_k z(0)^k=0$. Adding $M\boldsymbol{p}$ therefore preserves the published vector's constraint residuals instead of recomputing its central value from rounded $a_1$–$a_4$.

In [ ]:
eigenvalues, eigenvectors = np.linalg.eigh(minerva_a_cov_matrix)
order = np.argsort(eigenvalues)[::-1]
eigenvalues = np.maximum(eigenvalues[order], 0.0)
eigenvectors = eigenvectors[:, order]
B = eigenvectors * np.sqrt(eigenvalues)  # delta(a1..a4) = B @ p

partial_cv = np.asarray(minerva_a_values[1:5], dtype=float)
guess = minerva_a_values[0:1] + minerva_a_values[5:]
completed_rounded_cv = np.asarray(complete_a_values_8(partial_cv, minerva_t0, guess))
published_cv = np.asarray(minerva_a_values, dtype=float)
full_matrix = np.column_stack([
    np.asarray(complete_a_values_8(partial_cv + B[:, j], minerva_t0, guess)) - completed_rounded_cv
    for j in range(4)
])

assert np.allclose(B @ B.T, minerva_a_cov_matrix)
assert np.allclose(full_matrix[1:5], B)
minerva_full_covariance = full_matrix @ full_matrix.T
minerva_prior_sigma = np.sqrt(np.diag(minerva_full_covariance))

# Draw the four free coefficients from MINERvA's reported Gaussian covariance,
# then propagate their displacements to all nine coefficients.
free_to_full = np.linalg.solve(B.T, full_matrix.T).T  # S = M @ B^{-1}

prior_rng = np.random.default_rng(2026)
minerva_free_samples = prior_rng.multivariate_normal(mean=partial_cv, cov=minerva_a_cov_matrix, size=N_PRIOR_SAMPLES)
minerva_prior_samples = published_cv + (minerva_free_samples - partial_cv) @ free_to_full.T
pd.DataFrame(B, index=[f'a{i}' for i in range(1, 5)], columns=PCA_LABELS)

## Load and transform the joint posterior

In [ ]:
with uproot.open(ROOT_FILE) as root:
    chain_names = [k.split(';')[0] for k in root.keys() if k.split(';')[0].endswith('_mcmc_chain')]
    if len(chain_names) != 1:
        raise RuntimeError(f'Expected one MCMC chain, found {chain_names}')
    tree = root[chain_names[0]]
    pca_samples = np.column_stack([tree[name].array(library='np') for name in PCA_BRANCHES])
    fit_hist = root['global_fit_result']
    fit_labels = list(fit_hist.axis().labels())
    fit_values = fit_hist.values()
    pca_profile = np.asarray([fit_values[fit_labels.index(label)] for label in PCA_LABELS])

pca_samples = pca_samples[BURN_IN::THIN]
a_samples = published_cv + pca_samples @ full_matrix.T
a_profile = published_cv + full_matrix @ pca_profile
print(f'Transformed {len(a_samples):,} joint posterior samples')

## Posterior summary

Each row describes one z-expansion coefficient:

- **published_MINERvA**: the original nine-value MINERvA coefficient vector, used here as the exact PCA prior at $p=0$. The displayed coefficients are rounded, so their small normalization and sum-rule residuals are retained; every PCA displacement preserves those residuals.
- **MINERvA_prior_sigma**: the marginal one-standard-deviation width of the MINERvA prior. For $a_1$–$a_4$ this is obtained directly from the published covariance. For the dependent coefficients it is induced through the constraint-preserving transformation: $C_{\mathrm{full}}=MM^T$ and $\sigma_i=\sqrt{(C_{\mathrm{full}})_{ii}}$.
- **profile_best_fit**: the coefficient at the joint minimum of the profiled likelihood (using the PCA values stored in `global_fit_result`). Profiling minimizes over the other parameters rather than integrating over them. In this file all best-fit PCA knobs are zero, so this equals `published_MINERvA`.
- **posterior_mean**: the arithmetic average over the transformed joint MCMC samples. It uses the full posterior and is sensitive to asymmetric tails.
- **posterior_median**: the 50th percentile of the marginal MCMC posterior. Half the samples lie on either side; it need not equal the mean or profile best fit.
- **q16**, **q84**: the 16th and 84th percentiles, forming the central 68% marginal credible interval.
- **minus_1sigma**, **plus_1sigma**: the distances from the posterior median to `q16` and `q84`. They are separate because the posterior can be asymmetric.

In [ ]:
q16, median, q84 = np.quantile(a_samples, [0.16, 0.50, 0.84], axis=0)
summary = pd.DataFrame({
    'published_MINERvA': np.asarray(minerva_a_values),
    'MINERvA_prior_sigma': minerva_prior_sigma,
    'profile_best_fit': a_profile,
    'posterior_mean': a_samples.mean(axis=0),
    'posterior_median': median,
    'q16': q16, 'q84': q84,
    'minus_1sigma': median - q16,
    'plus_1sigma': q84 - median,
}, index=[f'a{i}' for i in range(9)])
summary.index.name = 'coefficient'
summary

## Visualize the transformed marginals

The red step histogram is the MINERvA prior obtained by drawing $(a_1,\ldots,a_4)$ from the multivariate Gaussian defined by the published $4\times4$ covariance matrix and propagating each draw through the linear constraint-preserving map. This Gaussian is an approximation: a covariance matrix alone does not specify non-Gaussian likelihood structure. Under this approximation, every transformed marginal is Gaussian because the transformation is linear.

Rounding does not change the widths or Gaussian shapes. The published nine coefficients define the center, while the sampled changes are homogeneous constraint-preserving displacements. Consequently, all samples retain the same small constraint residuals as the rounded published central vector.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 9))
for i, ax in enumerate(axes.flat):
    bins = np.histogram_bin_edges(
        np.concatenate([minerva_prior_samples[:, i], a_samples[:, i]]), bins=50
    )
    ax.hist(minerva_prior_samples[:, i], bins=bins, density=True, histtype='step',
            color='C3', linewidth=1.5, label='MINERvA prior distribution')
    ax.hist(a_samples[:, i], bins=bins, density=True, histtype='stepfilled',
            color='C0', alpha=0.40, label='Post-fit posterior distribution')
    ax.axvline(a_profile[i], color='k', linestyle='--', label='Best fit')
    ax.axvspan(q16[i], q84[i], color='C1', alpha=0.20,
               label=r'Post-fit $68\%$ interval')

    ax.axvline(minerva_a_values[i], color='C3', linestyle=':', linewidth=2,
               label='Published MINERvA prior')
    ax.axvspan(minerva_a_values[i] - minerva_prior_sigma[i],
               minerva_a_values[i] + minerva_prior_sigma[i],
               color='C3', alpha=0.12, label=r'Prior $1\sigma$')

    ax.set_xlabel(f'$a_{i}$')

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.02), ncol=3, fontsize=10, frameon=False)
fig.supylabel('Probability density')
fig.tight_layout(rect=[0, 0, 1, 0.96])

In [ ]:
# Compare the prior and post-fit point estimates and one-sigma intervals.
fig, axes = plt.subplots(3, 3, figsize=(12, 7.5))
for i, ax in enumerate(axes.flat):
    prior_central = published_cv[i]
    prior_low = prior_central - minerva_prior_sigma[i]
    prior_high = prior_central + minerva_prior_sigma[i]

    ax.hlines(1, prior_low, prior_high, color='C3', linewidth=3)
    ax.plot(prior_central, 1, 'o', color='C3', markersize=6,
            label=r'Prior central $\pm 1\sigma$')
    ax.hlines(0, q16[i], q84[i], color='C0', linewidth=3)
    ax.plot(a_profile[i], 0, 'D', color='C0', markersize=6,
            label=r'Post-fit best fit and central $68\%$ interval')

    ax.set_yticks([0, 1], ['Post-fit', 'Prior'])
    ax.set_xlabel(f'$a_{i}$')
    ax.grid(axis='x', alpha=0.25)
    ax.set_ylim(-0.6, 1.6)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2, frameon=False)
fig.suptitle(r'Prior and post-fit best fits with $1\sigma$ uncertainties', y=0.94)
fig.tight_layout(rect=[0, 0, 1, 0.95])